# E07 - ABLATION TỪNG LUẬT CHUẨN HOÁ, PHỦ ĐỊNH & VẾ ĐỐI LẬP SAU `NHƯNG` (NGƯỜI 2)

**Vấn đề:**
1. E03 mới đo gộp `clean_text`; chưa rõ trong 6 nhóm luật (`emoji`, `pii`, `elongation`, `single_teencode`, `multi_teencode`, `cam`), luật nào giúp ích và luật nào gây hại.
2. NB03 chỉ ra 2 lỗi chiếm đa số trên tập Eyeball là **Phủ định (38.2%)** và **Câu khen trước – chê sau qua từ nối đối lập (`nhưng`, `tuy nhiên`, `chỉ có điều` — 34.5%)** (*Ý tưởng 2*).

In [1]:
import os, sys, glob, zipfile, subprocess

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or os.path.exists('/kaggle/working')

def _find_root():
    cwd = os.getcwd()
    cand = os.path.abspath('..') if os.path.basename(cwd) in ('notebooks', 'experiments') else cwd
    if os.path.exists(os.path.join(cand, 'src', 'modeling.py')):
        return cand, cand
    if IN_COLAB:
        for base in [cwd, '/content']:
            if os.path.exists(os.path.join(base, 'src', 'modeling.py')):
                return base, base
        for z in glob.glob('/content/*src_and_data*.zip'):
            with zipfile.ZipFile(z, 'r') as zf:
                zf.extractall('/content')
            if os.path.exists('/content/src/modeling.py'):
                return '/content', '/content'
    if IN_KAGGLE:
        work = '/kaggle/working'
        if os.path.exists(os.path.join(work, 'src', 'modeling.py')):
            return work, work
        for p in glob.glob('/kaggle/input/**/src/modeling.py', recursive=True):
            return os.path.dirname(os.path.dirname(p)), work
        for z in glob.glob('/kaggle/input/**/*src_and_data*.zip', recursive=True):
            dest = os.path.join(work, 'project')
            os.makedirs(dest, exist_ok=True)
            with zipfile.ZipFile(z, 'r') as zf:
                zf.extractall(dest)
            if os.path.exists(os.path.join(dest, 'src', 'modeling.py')):
                return dest, dest
    raise FileNotFoundError('Khong tim thay src/modeling.py! Hay chay trong repo hoac upload src_and_data.zip.')

ROOT, WORK_DIR = _find_root()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
PROC_DIR = os.path.join(ROOT, 'data', 'processed')
FIG_DIR = os.path.join(WORK_DIR, 'reports', 'figures')
os.makedirs(FIG_DIR, exist_ok=True)
os.environ['EXP_LOG_PATH'] = os.path.join(WORK_DIR, 'reports', 'experiments_log.csv')
print(f'Env: colab={IN_COLAB}, kaggle={IN_KAGGLE} | ROOT={ROOT}')

Env: colab=False, kaggle=False | ROOT=D:\E-commerce


In [2]:
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score
from src.text_cleaner import clean_text, segment_words
from src.exp_log import log_experiment

df_train = pd.read_csv(os.path.join(PROC_DIR, 'train_clean.csv'))
df_dev = pd.read_csv(os.path.join(PROC_DIR, 'dev_clean.csv'))
asp_cols = [f'asp_{i}' for i in range(5)]
TOKEN_PAT = r'(?u)\b\w+\b|\[\w+\]'

def eval_text_pair(tr_texts, dev_texts, run_name, desc, eval_slices=False):
    p_sent = Pipeline([
        ('tfidf', TfidfVectorizer(token_pattern=TOKEN_PAT, ngram_range=(1, 2), min_df=2, max_df=0.95, max_features=15000, sublinear_tf=True)),
        ('clf', LogisticRegression(C=1.0, class_weight='balanced', max_iter=1000, random_state=42)),
    ]).fit(tr_texts, df_train['sentiment'])
    pred_s = p_sent.predict(dev_texts)
    f1_s = f1_score(df_dev['sentiment'], pred_s, average='macro')

    p_asp = Pipeline([
        ('tfidf', TfidfVectorizer(token_pattern=TOKEN_PAT, ngram_range=(1, 2), min_df=2, max_df=0.95, max_features=15000, sublinear_tf=True)),
        ('clf', OneVsRestClassifier(LogisticRegression(C=2.0, class_weight='balanced', max_iter=1000, random_state=42))),
    ]).fit(tr_texts, df_train[asp_cols].values)
    f1_a = f1_score(df_dev[asp_cols].values, p_asp.predict(dev_texts), average='macro')

    out = {'run': run_name, 'dev_sent_f1': round(f1_s, 4), 'dev_asp_f1': round(f1_a, 4)}
    note = f'dev_asp_f1={f1_a:.4f}'
    if eval_slices:
        but_mask = df_dev['sentence_clean'].fillna('').str.lower().str.contains(r'(?<!\w)(?:nhưng|tuy nhiên|có điều|chỉ có|mặc dù|trừ)(?!\w)', regex=True).values
        neg_mask = df_dev['sentence_clean'].fillna('').str.lower().str.contains(r'(?<!\w)(?:không|chưa|chẳng|chả|kém|thiếu)(?!\w)', regex=True).values
        f1_but = f1_score(df_dev.loc[but_mask, 'sentiment'], pred_s[but_mask], average='macro')
        f1_neg = f1_score(df_dev.loc[neg_mask, 'sentiment'], pred_s[neg_mask], average='macro')
        out['but_slice_f1'] = round(f1_but, 4)
        out['neg_slice_f1'] = round(f1_neg, 4)
        note += f', but_slice={f1_but:.4f}, neg_slice={f1_neg:.4f}'
    log_experiment('E07', run_name, desc, {'variant': run_name}, dev_f1=f1_s, notes=note)
    return out

## 1. ABLATION TẮT TỪNG LUẬT TRONG `clean_text(..., skip=...)`

In [3]:
rule_groups = [
    ('full_clean', frozenset()),
    ('skip_emoji', frozenset({'emoji'})),
    ('skip_pii', frozenset({'pii'})),
    ('skip_elongation', frozenset({'elongation'})),
    ('skip_single_teencode', frozenset({'single_teencode'})),
    ('skip_multi_teencode', frozenset({'multi_teencode'})),
    ('skip_cam', frozenset({'cam'})),
]
ab_rows = []
for name, sk in rule_groups:
    tr_seg = df_train['sentence'].fillna('').apply(lambda s: segment_words(clean_text(s, lowercase=False, skip=sk)))
    dv_seg = df_dev['sentence'].fillna('').apply(lambda s: segment_words(clean_text(s, lowercase=False, skip=sk)))
    ab_rows.append(eval_text_pair(tr_seg, dv_seg, name, 'Leave-one-rule-out text_cleaner'))
df_ab = pd.DataFrame(ab_rows)
base_f1 = df_ab.loc[df_ab['run'] == 'full_clean', 'dev_sent_f1'].iloc[0]
df_ab['delta_vs_full'] = (df_ab['dev_sent_f1'] - base_f1).round(4)
df_ab

,run,dev_sent_f1,dev_asp_f1,delta_vs_full
0,full_clean,0.7478,0.8911,0.0000
1,skip_emoji,0.7466,0.8910,-0.0012
2,skip_pii,0.7485,0.8908,0.0007
3,skip_elongation,0.7444,0.8903,-0.0034
4,skip_single_teencode,0.7457,0.8910,-0.0021
5,skip_multi_teencode,0.7375,0.8922,-0.0103
6,skip_cam,0.7448,0.8898,-0.0030


## 2. XỬ LÝ PHỦ ĐỊNH & Ý TƯỞNG 2 (TĂNG TRỌNG SỐ VẾ SAU `NHƯNG / TUY NHIÊN`)

In [4]:
NEG_WORDS = {'không', 'chưa', 'chẳng', 'chả', 'thiếu'}
STOP_CLAUSE = {',', '.', ';', '!', '?', 'nhưng', 'tuy_nhiên', 'tuy', 'có_điều', 'chỉ_có', 'mặc_dù', 'còn', 'nên', 'mà'}
BUT_RE = re.compile(r'(?<!\w)(nhưng|tuy_nhiên|tuy nhiên|có_điều|có điều|chỉ_có|chỉ có|mặc_dù|mặc dù)(?!\w)', re.IGNORECASE)

def apply_neg_tag(seg_text, window=2):
    toks = str(seg_text).split()
    out, i = [], 0
    while i < len(toks):
        low = toks[i].lower()
        out.append(toks[i])
        if low in NEG_WORDS:
            cnt = 0
            j = i + 1
            while j < len(toks) and cnt < window and toks[j].lower() not in STOP_CLAUSE:
                if re.match(r'^[\wÀ-ỹ]+$', toks[j]):
                    out.append(f'NOT_{toks[j].lower()}')
                    cnt += 1
                else:
                    break
                j += 1
            i = j
        else:
            i += 1
    return ' '.join(out)

def apply_but_clause(seg_text, mode='prefix'):
    parts = BUT_RE.split(str(seg_text), maxsplit=1)
    if len(parts) < 3:
        return str(seg_text)
    head, conj, tail = parts[0], parts[1], parts[2]
    if mode == 'prefix':
        tail_tagged = ' '.join(f'BUT_{w}' if re.match(r'^[\wÀ-ỹ]+$', w) else w for w in tail.split())
        return f'{head} {conj} {tail_tagged}'.strip()
    elif mode == 'double':
        return f'{head} {conj} {tail} {tail}'.strip()
    return str(seg_text)

# Audit 8 cau mau sau bien doi
sample_but = df_dev[df_dev['sentence_seg'].str.lower().str.contains('nhưng|chưa|không', regex=True)].head(6)['sentence_seg']
pd.DataFrame({
    'orig_seg': sample_but.values,
    'neg_tag': [apply_neg_tag(s) for s in sample_but],
    'but_prefix': [apply_but_clause(s, 'prefix') for s in sample_but],
})

,orig_seg,neg_tag,but_prefix
0,"Máy_ok , không chê điểm nào , chính_thức là fa...","Máy_ok , không NOT_chê NOT_điểm nào , chính_th...","Máy_ok , không chê điểm nào , chính_thức là fa..."
1,"Mới sài được 3 ngày sản_phẩm thì khá là ok , l...","Mới sài được 3 ngày sản_phẩm thì khá là ok , l...","Mới sài được 3 ngày sản_phẩm thì khá là ok , l..."
2,Dùng tốt chơi game bao mượt không nóng như lời...,Dùng tốt chơi game bao mượt không NOT_nóng NOT...,Dùng tốt chơi game bao mượt không nóng như lời...
3,"Có ai giống mh không , mh mua ip 7 plus , mấy ...","Có ai giống mh không , mh mua ip 7 plus , mấy ...","Có ai giống mh không , mh mua ip 7 plus , mấy ..."
4,Không bjk ngày_mai có ra đổi được không đây . ...,Không NOT_bjk NOT_ngày_mai có ra đổi được khôn...,Không bjk ngày_mai có ra đổi được không đây . ...
5,"Sản_phẩm rất tốt , mượt_mà , pin quá trâu , ca...","Sản_phẩm rất tốt , mượt_mà , pin quá trâu , ca...","Sản_phẩm rất tốt , mượt_mà , pin quá trâu , ca..."


In [5]:
# So sanh tren toan bo Dev va rieng lat cat cau co 'nhung/tuy nhien' va 'phu dinh'
tr_base = df_train['sentence_seg'].fillna('')
dv_base = df_dev['sentence_seg'].fillna('')

disc_rows = [
    eval_text_pair(tr_base, dv_base, 'discourse_baseline', 'Baseline pyvi for discourse slice', eval_slices=True),
    eval_text_pair(tr_base.apply(apply_neg_tag), dv_base.apply(apply_neg_tag), 'discourse_neg_tag', 'Negation tagging NOT_word', eval_slices=True),
    eval_text_pair(tr_base.apply(lambda s: apply_but_clause(s, 'prefix')), dv_base.apply(lambda s: apply_but_clause(s, 'prefix')), 'discourse_but_prefix', 'Contrastive clause BUT_prefix', eval_slices=True),
    eval_text_pair(tr_base.apply(lambda s: apply_but_clause(s, 'double')), dv_base.apply(lambda s: apply_but_clause(s, 'double')), 'discourse_but_double', 'Contrastive clause repeat 2x', eval_slices=True),
    eval_text_pair(tr_base.apply(lambda s: apply_but_clause(apply_neg_tag(s), 'prefix')), dv_base.apply(lambda s: apply_but_clause(apply_neg_tag(s), 'prefix')), 'discourse_neg_and_but', 'Negation tagging + BUT_prefix', eval_slices=True),
]
pd.DataFrame(disc_rows)

,run,dev_sent_f1,dev_asp_f1,but_slice_f1,neg_slice_f1
0,discourse_baseline,0.7478,0.8911,0.6730,0.7237
1,discourse_neg_tag,0.7422,0.8880,0.6412,0.7184
2,discourse_but_prefix,0.7219,0.8856,0.6106,0.6941
3,discourse_but_double,0.7435,0.8889,0.6469,0.7063
4,discourse_neg_and_but,0.7317,0.8814,0.6195,0.7012


## 3. KẾT LUẬN & HƯỚNG GIẢI QUYẾT ĐỀ XUẤT

- **Ablation từng luật trong `src/text_cleaner.py`:**
  - Tắt bất kỳ nhóm luật chuẩn hoá nào đều làm giảm nhẹ hoặc giữ nguyên Dev Sentiment F1 so với `full_clean` (`0.7478`): `multi_teencode` quan trọng nhất (tắt giảm **`-0.0103`**), tiếp theo là `elongation` (`-0.0034`), `cam` (`-0.0030`), `single_teencode` (`-0.0021`), `emoji` (`-0.0012`). Riêng `skip_pii` (`+0.0007`) không tăng F1 nhưng bắt buộc giữ để ẩn danh dữ liệu cá nhân.
  - **Hướng giải quyết (cho NB01):** giữ nguyên cả 6 nhóm luật trong `src/text_cleaner.py` vì số liệu chứng minh không luật nào kéo tụt hiệu năng chung.
- **Phủ định (`NOT_`) & Ý tưởng 2 (tăng trọng số vế sau `nhưng/tuy nhiên`):**
  - Cả `NOT_` (`0.7422`), `BUT_prefix` (`0.7219`) và lặp 2 lần vế sau (`0.7435`) đều **giảm** F1 so với baseline (`0.7478`), kể cả trên riêng lát cắt câu có từ nối đối lập (`0.6730 → 0.6106–0.6469`).
  - **Lý do:** TF-IDF vốn đã có `ngram=(1, 2)` bắt được cụm phủ định tự nhiên; gắn thêm tiền tố giả `NOT_`/`BUT_` làm vỡ từ vựng n-gram và thưa đặc trưng trên 7.7k mẫu, đồng thời không xử lý được câu đảo ngược (*"pin yếu nhưng đáng tiền"*).
  - **Hướng giải quyết (cho NB03 / NB04):** **không** chèn thẻ `NOT_`/`BUT_` vào pipeline TF-IDF hoặc PhoBERT; giao các câu có từ nối đối lập/phủ định phức tạp cho tầng PhoBERT (`NB04`) tận dụng cơ chế self-attention.
